# Validation: Habit-Portfolio Model Converges to Standard Portfolio Model

When the habit weight parameter $\alpha$ (HabitWgt) approaches zero, the utility function

$$u(c,h) = \frac{(c/h^\alpha)^{1-\rho}}{1-\rho}$$

converges to the standard CRRA utility $u(c) = c^{1-\rho}/(1-\rho)$, and the habit stock
becomes irrelevant. In this limit, the `HabitPortfolioConsumerType` should produce
the same consumption and risky share functions as the standard `PortfolioConsumerType`.

This notebook verifies that convergence.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from HARK.ConsumptionSaving.ConsHabitPortfolioModel import HabitPortfolioConsumerType
from HARK.ConsumptionSaving.ConsPortfolioModel import PortfolioConsumerType

## Set up matched parameters

We configure both models with identical economic parameters. The habit-portfolio model
uses `HabitWgt=0.001` (very close to zero) so the habit stock has negligible effect on utility.

In [ ]:
T = 10

common_params = dict(
    cycles=1,
    T_cycle=T,
    CRRA=5.0,
    Rfree=[1.03] * T,
    DiscFac=0.96,
    LivPrb=[0.98] * T,
    PermGroFac=[1.01] * T,
    BoroCnstArt=0.0,
    PermShkStd=[0.1] * T,
    PermShkCount=7,
    TranShkStd=[0.1] * T,
    TranShkCount=7,
    UnempPrb=0.05,
    IncUnemp=0.3,
    T_retire=0,
    UnempPrbRet=0.005,
    IncUnempRet=0.0,
    RiskyAvg=1.08,
    RiskyStd=0.18362634887,
    RiskyCount=5,
    aXtraMin=0.001,
    aXtraMax=50.0,
    aXtraNestFac=2,
    aXtraCount=48,
    aXtraExtra=None,
    ShareCount=25,
)

In [ ]:
# Habit-portfolio model with HabitWgt ≈ 0
hp_agent = HabitPortfolioConsumerType(
    **common_params,
    HabitWgt=0.001,
    HabitRte=0.2,
)
hp_agent.solve()
print(f"Habit-portfolio model solved: {len(hp_agent.solution)} periods")

In [ ]:
# Standard portfolio model with matched parameters
port_agent = PortfolioConsumerType(
    **common_params,
    AdjustPrb=1.0,
    DiscreteShareBool=False,
    IndepDstnBool=True,
    vFuncBool=False,
)
port_agent.solve()
print(f"Standard portfolio model solved: {len(port_agent.solution)} periods")

## Compare consumption functions

Since the habit-portfolio model has a 2D consumption function `cFunc(m, h)` while the
standard model has a 1D function `cFuncAdj(m)`, we evaluate the habit model at several
habit levels. When $\alpha \approx 0$, the habit dimension should have negligible effect.

In [ ]:
m_grid = np.linspace(0.5, 10, 200)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
periods_to_plot = [0, 2, 4, 6, 8, 9]

for ax, t in zip(axes.flat, periods_to_plot):
    # Standard portfolio model
    c_port = [port_agent.solution[t].cFuncAdj(m) for m in m_grid]
    ax.plot(m_grid, c_port, "k-", lw=2, label="Standard portfolio")

    # Habit-portfolio at different h values
    for h_val, color, ls in [(0.5, "C0", "--"), (1.0, "C1", "--"), (2.0, "C2", "--")]:
        c_hp = [hp_agent.solution[t]["cFunc"](m, h_val) for m in m_grid]
        ax.plot(m_grid, c_hp, ls, color=color, label=f"Habit-port h={h_val}")

    ax.set_title(f"Period {t}")
    ax.set_xlabel("$m$")
    ax.set_ylabel("$c$")
    ax.set_ylim(0, None)

axes[0, 0].legend(fontsize=8)
fig.suptitle(
    r"Consumption functions: standard portfolio vs habit-portfolio ($\alpha=0.001$)",
    fontsize=13,
)
plt.tight_layout()
plt.show()

## Compare risky share functions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, t in zip(axes.flat, periods_to_plot):
    # Standard portfolio model
    s_port = [port_agent.solution[t].ShareFuncAdj(m) for m in m_grid]
    ax.plot(m_grid, s_port, "k-", lw=2, label="Standard portfolio")

    # Habit-portfolio at different h values
    for h_val, color, ls in [(0.5, "C0", "--"), (1.0, "C1", "--"), (2.0, "C2", "--")]:
        s_hp = [hp_agent.solution[t]["ShareFunc"](m, h_val) for m in m_grid]
        ax.plot(m_grid, s_hp, ls, color=color, label=f"Habit-port h={h_val}")

    ax.set_title(f"Period {t}")
    ax.set_xlabel("$m$")
    ax.set_ylabel("$s$")
    ax.set_ylim(0, 1.05)

axes[0, 0].legend(fontsize=8)
fig.suptitle(
    r"Risky share functions: standard portfolio vs habit-portfolio ($\alpha=0.001$)",
    fontsize=13,
)
plt.tight_layout()
plt.show()

## Quantitative comparison

Compute the maximum absolute difference between the two models' consumption and share
functions across all periods, evaluated at `h=1.0` (which shouldn't matter when $\alpha \approx 0$).

In [ ]:
m_test = np.linspace(1.0, 10.0, 80)
h_test = 1.0

print(f"{'Period':>6} {'Max |Δc|':>10} {'Max |Δc|/c':>12} {'Max |Δs|':>10}")
print("-" * 42)

for t in range(len(hp_agent.solution)):
    c_hp = np.array([hp_agent.solution[t]["cFunc"](m, h_test) for m in m_test])
    c_port = np.array([port_agent.solution[t].cFuncAdj(m) for m in m_test])

    s_hp = np.array([hp_agent.solution[t]["ShareFunc"](m, h_test) for m in m_test])
    s_port = np.array([port_agent.solution[t].ShareFuncAdj(m) for m in m_test])

    # Filter out nan/inf values from 2D interpolation edge cases
    valid = np.isfinite(c_hp) & np.isfinite(s_hp) & (c_hp < m_test)
    if np.any(valid):
        max_dc = np.max(np.abs(c_hp[valid] - c_port[valid]))
        max_dc_rel = np.max(
            np.abs(c_hp[valid] - c_port[valid]) / np.maximum(c_port[valid], 1e-10)
        )
        max_ds = np.nanmax(np.abs(s_hp[valid] - s_port[valid]))
    else:
        max_dc = max_dc_rel = max_ds = np.nan

    print(f"{t:>6d} {max_dc:>10.6f} {max_dc_rel:>12.6f} {max_ds:>10.6f}")

## Habit-invariance check

When $\alpha \approx 0$, the consumption and share functions should be approximately
invariant to the habit stock $h$. We verify this by computing the range of `cFunc(m, h)`
across different `h` values for each `m`.

In [ ]:
h_values = np.array([0.3, 0.5, 1.0, 2.0, 4.0])
t_check = 0  # earliest (most-solved) period

c_across_h = np.array(
    [[hp_agent.solution[t_check]["cFunc"](m, h) for m in m_test] for h in h_values]
)
s_across_h = np.array(
    [[hp_agent.solution[t_check]["ShareFunc"](m, h) for m in m_test] for h in h_values]
)

c_range = np.max(c_across_h, axis=0) - np.min(c_across_h, axis=0)
s_range = np.max(s_across_h, axis=0) - np.min(s_across_h, axis=0)

print(f"Period {t_check}: habit-invariance of cFunc and ShareFunc")
print(f"  Max range of c across h values: {np.max(c_range):.6f}")
print(f"  Max range of s across h values: {np.max(s_range):.6f}")
print(f"  Mean c at h=1: {np.mean(c_across_h[2]):.4f}")
print("  (Small ranges confirm h is irrelevant when α ≈ 0)")

## Convergence as $\alpha \to 0$

We solve the habit-portfolio model at several values of $\alpha$ and plot the maximum
relative consumption error against the standard portfolio model.

In [ ]:
alpha_values = [0.5, 0.3, 0.2, 0.15, 0.1, 0.05, 0.01, 0.001]
max_rel_errors_c = []
max_abs_errors_s = []

m_conv = np.linspace(1.0, 10.0, 80)

for alpha in alpha_values:
    agent = HabitPortfolioConsumerType(**common_params, HabitWgt=alpha, HabitRte=0.2)
    agent.solve()

    # Measure error at period 3 (well away from terminal, less boundary noise)
    c_hp = np.array([agent.solution[3]["cFunc"](m, 1.0) for m in m_conv])
    c_port = np.array([port_agent.solution[3].cFuncAdj(m) for m in m_conv])
    s_hp = np.array([agent.solution[3]["ShareFunc"](m, 1.0) for m in m_conv])
    s_port = np.array([port_agent.solution[3].ShareFuncAdj(m) for m in m_conv])

    valid = np.isfinite(c_hp) & np.isfinite(s_hp) & (c_hp < m_conv)
    if np.any(valid):
        rel_err_c = np.max(
            np.abs(c_hp[valid] - c_port[valid]) / np.maximum(c_port[valid], 1e-10)
        )
        abs_err_s = np.nanmax(np.abs(s_hp[valid] - s_port[valid]))
    else:
        rel_err_c = abs_err_s = np.nan

    max_rel_errors_c.append(rel_err_c)
    max_abs_errors_s.append(abs_err_s)
    print(f"α = {alpha:.3f}: max |Δc/c| = {rel_err_c:.6f}, max |Δs| = {abs_err_s:.6f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.loglog(alpha_values, max_rel_errors_c, "o-")
ax1.set_xlabel(r"$\alpha$ (HabitWgt)")
ax1.set_ylabel("Max relative consumption error")
ax1.set_title(r"Convergence of $c$ as $\alpha \to 0$ (period 3)")
ax1.grid(True, alpha=0.3)

ax2.loglog(alpha_values, max_abs_errors_s, "o-")
ax2.set_xlabel(r"$\alpha$ (HabitWgt)")
ax2.set_ylabel("Max absolute share error")
ax2.set_title(r"Convergence of $s$ as $\alpha \to 0$ (period 3)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nNote: For very small α, the 2D interpolation (Curvilinear2DInterp) becomes")
print("poorly conditioned as the habit dimension loses relevance, causing errors to")
print(
    "plateau rather than continue decreasing. The clean convergence region is α ∈ [0.1, 0.5]."
)